# Cross-fit SR-PDS on the non-confounding designs (DGP-1--5)

The cross-fit variant was run at the full replication count only on the three
confounding designs; the tables currently carry DGP-1--5 from an earlier run.
This fills them in at $n=500$, $n_{\text{rep}}=100$ (the same budget as the
confounding designs), so the SR-PDS\,(CF) row is uniform across all eight
columns and the ``carried from an earlier run'' caveat can be dropped.

These are easy designs ($m=0$ on 1--4, linear $m$ on 5), so the per-fold PySR
budget is the cheap configuration rather than the heavy severe-design one. The
run is **resumable** and writes to a **new** file; the existing
`cf_improved.pkl` / `cf_improved_foldrec.pkl` (DGP-6/7/8) are not touched.


## Setup


In [ ]:
import time
import numpy as np, pandas as pd
import config as C
from dgp import DGP_REGISTRY
from cf_parallel import run_cf_parallel

N_JOBS_CF = 4            # 5 GB free -> 4 workers
N_REPS_CF = C.N_REPS_CF  # 100, matching the confounding-design CF run
EASY = ['dgp1','dgp2','dgp3','dgp4','dgp5']

# cheap per-fold budget: these designs have no nonlinear confounder to chase
CF_CFG_EASY = {'n_folds': 5, 'niterations': 40, 'parsimony_d': 0.01}

CF_PKL = C.RESULTS_DIR / 'cf_easy_dgp1_5.pkl'   # NEW file; originals untouched
print('writes to:', CF_PKL.name, '| n_rep =', N_REPS_CF, '| jobs =', N_JOBS_CF)


## Run (per-DGP, resumable)

Same parallel cross-fit driver used for the confounding designs, so the
estimator and its influence-function variance are identical; only the design
and the (cheaper) per-fold budget differ.


In [ ]:
if CF_PKL.exists():
    prev = pd.read_pickle(CF_PKL); done = set(prev['dgp'].unique()); all_reps = [prev]
    print('resuming; done:', sorted(done))
else:
    done, all_reps = set(), []

t0 = time.time()
for dgp_key in EASY:
    if dgp_key in done:
        continue
    print(f'\n=== {dgp_key} ({DGP_REGISTRY[dgp_key]["label"]}) ===', flush=True)
    df = run_cf_parallel(DGP_REGISTRY[dgp_key]['fn'], n_reps=N_REPS_CF,
                         n=C.N_HEADLINE, p=C.P, s=C.S, beta0=C.BETA0,
                         cf_config=CF_CFG_EASY, n_jobs=N_JOBS_CF, desc=dgp_key)
    df['dgp']=dgp_key; df['estimator']='sr_pds_cf'
    df['dgp_label']=DGP_REGISTRY[dgp_key]['label']; df['est_label']='SR-PDS (CF)'
    df['beta0']=C.BETA0
    all_reps.append(df)
    pd.concat(all_reps, ignore_index=True).to_pickle(CF_PKL)
    ok = df[~df['failed']]
    cov = ((ok['ci_low'] < C.BETA0) & (C.BETA0 < ok['ci_high'])).mean()
    print(f'  coverage={cov:.3f}  [{(time.time()-t0)/60:.1f} min]', flush=True)
print('\nCF on DGP-1--5 complete -> cf_easy_dgp1_5.pkl')


## Reduce to bias / RMSE / coverage

Same `evaluate` reducer as every other method; these five rows replace the
carried-over DGP-1--5 entries in the SR-PDS\,(CF) row of Tables 3--5.


In [ ]:
from evaluate import evaluate
cf = pd.read_pickle(CF_PKL)
out = []
for dgp_key, g in cf.groupby('dgp'):
    ok = g[~g['failed']]
    m = evaluate(ok, C.BETA0)
    m.update({'dgp': dgp_key, 'dgp_label': DGP_REGISTRY[dgp_key]['label'],
              'estimator': 'sr_pds_cf', 'est_label': 'SR-PDS (CF)',
              'n': C.N_HEADLINE, 'n_rep': int(len(ok))})
    out.append(m)
cf_easy = pd.DataFrame(out).sort_values('dgp')
cf_easy.to_csv(C.RESULTS_DIR / 'cf_easy_dgp1_5.csv', index=False)
print('wrote cf_easy_dgp1_5.csv\n')
print(cf_easy[['dgp','dgp_label','bias','rmse','coverage','n_rep']].to_string(index=False))


## Done

`cf_easy_dgp1_5.csv` holds the SR-PDS\,(CF) bias / RMSE / coverage for the five
non-confounding designs at $n_{\text{rep}}=100$. With the DGP-6/7/8 CF results
already in hand, the SR-PDS\,(CF) row is now uniform at $n_{\text{rep}}=100$
across all eight designs, and the ``DGP-1--5 carried from an earlier run''
clause can be removed from the three table captions.
